# LPRec3D: original scaling and padding regressions

These cases complement the current worked examples. They retain the original preprocessing, detector selections, method arguments, numerical references and tolerances from the remaining Zenodo tests. In particular, LPRec keeps the default legacy scaling and FISTA/ADMM keep the original regularisation settings. They must not be confused with examples that deliberately use different scaling or parameters. Each original assertion is recorded separately and enforced by the final summary; a mismatch remains a failing regression.

Transferred from `zenodo-tests/test_recon/test_algorithm.py` (SHA-256 `6bc5eaa150a5ec20cd8343b7bb26bcabaff6424a8e5d81964e727aad90266189`).
Data: [Zenodo record 17252190](https://zenodo.org/records/17252190).
The original numerical references and tolerances are retained. Figures support visual review;
regression checks are evaluated after all cases so failures retain their figures and measurements.

## Execution

Install `.[validation]` and download the archives with `.scripts/download_zenodo.py Zenodo`.
Set `HTTOMOLIBGPU_DATA_DIR` to use an existing archive directory. Run GPU notebooks serially:

```bash
python -m pytest --nbmake --nbmake-timeout=1800 --overwrite docs/source/validation/reconstruction/transferred/lprec_legacy.ipynb
```

For a conda CUDA installation, set `CUDA_PATH` to the toolkit root containing
`include/cuda_runtime.h` if CuPy cannot locate the headers. Documentation renders
saved outputs without executing GPU computations.

## Environment and archive loading


In [ ]:
import gc
import hashlib
import json
import os
import platform
from importlib.metadata import version
from pathlib import Path
from math import isclose

import cupy as cp
import matplotlib.pyplot as plt
import numpy as np
import pytest
from numpy.testing import assert_allclose

from httomolibgpu.prep.normalize import dark_flat_field_correction, minus_log

repo_root = next(
    (
        p
        for p in (Path.cwd(), *Path.cwd().parents)
        if (p / "pyproject.toml").is_file() and (p / "httomolibgpu").is_dir()
    ),
    None,
)
configured_dir = os.environ.get("HTTOMOLIBGPU_DATA_DIR")
if configured_dir:
    data_dir = Path(configured_dir).expanduser()
elif repo_root is not None:
    data_dir = repo_root / "Zenodo"
else:
    raise FileNotFoundError("Set HTTOMOLIBGPU_DATA_DIR to the Zenodo archive directory")
props = cp.cuda.runtime.getDeviceProperties(cp.cuda.Device().id)
print(
    json.dumps(
        {
            "python": platform.python_version(),
            "packages": {
                name: version(name)
                for name in (
                    "httomolibgpu",
                    "tomobar",
                    "astra-toolbox",
                    "numpy",
                    "matplotlib",
                )
            },
            "cupy": cp.__version__,
            "cuda_runtime": cp.cuda.runtime.runtimeGetVersion(),
            "cuda_driver": cp.cuda.runtime.driverGetVersion(),
            "gpu": (
                props["name"].decode()
                if isinstance(props["name"], bytes)
                else props["name"]
            ),
        },
        indent=2,
    )
)
results = []


def force_clean_gpu_memory():
    gc.collect()
    cp.get_default_memory_pool().free_all_blocks()
    cp.get_default_pinned_memory_pool().free_all_blocks()


def load_dataset(name):
    path = data_dir / (name + ".npz")
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing {path}; run .scripts/download_zenodo.py first"
        )
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    print(
        json.dumps(
            {
                "dataset": path.name,
                "zenodo_record": "17252190",
                "sha256": digest.hexdigest(),
            },
            indent=2,
        )
    )
    with np.load(path) as archive:
        if name == "i12_dataset3":
            return tuple(
                cp.asarray(archive[key]) for key in ("proj1", "proj2", "flats", "darks")
            )
        return (
            cp.asarray(archive["projdata"]),
            archive["angles"].copy(),
            cp.asarray(archive["flats"]),
            cp.asarray(archive["darks"]),
        )


def plot_comparison(before, after, title, label="Negative log transmission"):
    before, after = cp.asnumpy(before), cp.asnumpy(after)
    residual = before - after
    lo, hi = np.percentile(before[np.isfinite(before)], [1, 99])
    limit = float(np.percentile(np.abs(residual[np.isfinite(residual)]), 99)) or 1
    fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
    for ax, image, name, cmap, vmin, vmax in (
        (axes[0], before, "Before", "gray", lo, hi),
        (axes[1], after, "After", "gray", lo, hi),
        (axes[2], residual, "Removed component", "coolwarm", -limit, limit),
    ):
        im = ax.imshow(image, cmap=cmap, aspect="auto", vmin=vmin, vmax=vmax)
        ax.set(title=name, xlabel="Detector column", ylabel="Projection index")
        fig.colorbar(im, ax=ax, label=label)
    fig.suptitle(title)
    plt.show()
    plt.close(fig)


def check_results():
    print(json.dumps(results, indent=2))
    failed = [row for row in results if not row["passed"]]
    assert not failed, "Transferred regressions failed: " + json.dumps(failed, indent=2)
    print(f"All {len(results)} transferred cases passed.")

In [ ]:
from httomolibgpu.recon.algorithm import (
    FBP3d_tomobar,
    LPRec3d_tomobar,
    FISTA3d_tomobar,
    ADMM3d_tomobar,
)
from httomolibgpu.misc.morph import sino_360_to_180

current_case = None


def record_check(expression, passed):
    results.append(
        {"test": current_case, "assertion": expression, "passed": bool(passed)}
    )


def check_allclose(*args, **kwargs):
    try:
        assert_allclose(*args, **kwargs)
    except AssertionError as exc:
        results.append(
            {
                "test": current_case,
                "assertion": "assert_allclose",
                "observed": float(args[0]),
                "expected": float(args[1]),
                "tolerances": kwargs,
                "passed": False,
                "details": str(exc),
            }
        )
    else:
        results.append(
            {
                "test": current_case,
                "assertion": "assert_allclose",
                "observed": float(args[0]),
                "expected": float(args[1]),
                "tolerances": kwargs,
                "passed": True,
            }
        )


def report_reconstruction(recon_data):
    print(
        json.dumps(
            {
                "test": current_case,
                "shape": list(recon_data.shape),
                "dtype": str(recon_data.dtype),
                "sum_float32": float(np.sum(recon_data)),
                "minimum": float(np.min(recon_data)),
                "maximum": float(np.max(recon_data)),
            },
            indent=2,
        )
    )
    image = recon_data[:, recon_data.shape[1] // 2, :]
    lo, hi = np.percentile(image, [1, 99])
    fig, axes = plt.subplots(1, 2, figsize=(11, 5), constrained_layout=True)
    width = image.shape[0]
    for ax, view, title in (
        (axes[0], image[::2, ::2], "Middle reconstructed slice"),
        (
            axes[1],
            image[width // 4 : 3 * width // 4, width // 4 : 3 * width // 4],
            "Central crop",
        ),
    ):
        im = ax.imshow(view, cmap="gray", vmin=lo, vmax=hi)
        ax.set(
            title=title,
            xlabel="Slice column (display pixel)",
            ylabel="Slice row (display pixel)",
        )
        fig.colorbar(im, ax=ax, label="Reconstructed value (legacy scaling)")
    fig.suptitle(current_case)
    plt.show()
    plt.close(fig)

## LPRec3d_tomobar_i12_dataset1

Original case: `test_reconstruct_LPRec3d_tomobar_i12_dataset1`.

Method call (original arguments):

```python
LPRec3d_tomobar(data_normalised_cut, np.deg2rad(angles), center=1253.75, detector_pad=0, filter_type='shepp', filter_freq_cutoff=1.0, recon_mask_radius=2.0)
```

The code below retains the original computation and records every assertion before displaying the reconstruction.


In [ ]:
def test_reconstruct_LPRec3d_tomobar_i12_dataset1(i12_dataset1: tuple):
    force_clean_gpu_memory()
    projdata = i12_dataset1[0]
    angles = i12_dataset1[1]
    flats = i12_dataset1[2]
    darks = i12_dataset1[3]
    del i12_dataset1
    data_normalised = dark_flat_field_correction(projdata, flats, darks)
    data_normalised = minus_log(data_normalised)
    data_normalised_cut = data_normalised[:, 5:8, :]
    del flats, darks, projdata, data_normalised
    force_clean_gpu_memory()
    recon_data = LPRec3d_tomobar(
        data_normalised_cut,
        np.deg2rad(angles),
        center=1253.75,
        detector_pad=0,
        filter_type="shepp",
        filter_freq_cutoff=1.0,
        recon_mask_radius=2.0,
    )
    record_check("recon_data.flags.c_contiguous", recon_data.flags.c_contiguous)
    recon_data = cp.asnumpy(recon_data)
    record_check(
        "isclose(np.sum(recon_data), 9628.824, abs_tol=10 ** (-3))",
        isclose(np.sum(recon_data), 9628.824, abs_tol=10 ** (-3)),
    )
    record_check(
        "pytest.approx(np.max(recon_data), rel=0.001) == 0.006367563270032406",
        pytest.approx(np.max(recon_data), rel=0.001) == 0.006367563270032406,
    )
    record_check(
        "pytest.approx(np.min(recon_data), rel=0.001) == -0.0062076798",
        pytest.approx(np.min(recon_data), rel=0.001) == -0.0062076798,
    )
    record_check("recon_data.dtype == np.float32", recon_data.dtype == np.float32)
    record_check(
        "recon_data.shape == (2560, 3, 2560)", recon_data.shape == (2560, 3, 2560)
    )
    report_reconstruction(recon_data)

In [ ]:
current_case = "test_reconstruct_LPRec3d_tomobar_i12_dataset1"
test_reconstruct_LPRec3d_tomobar_i12_dataset1(load_dataset("i12_dataset1"))
force_clean_gpu_memory()
print(json.dumps([row for row in results if row["test"] == current_case], indent=2))

## LPRec3d_tomobar_i12_dataset1_autopad

Original case: `test_reconstruct_LPRec3d_tomobar_i12_dataset1_autopad`.

Method call (original arguments):

```python
LPRec3d_tomobar(data_normalised_cut, np.deg2rad(angles), center=1253.75, detector_pad=True, filter_type='shepp', filter_freq_cutoff=1.0, recon_mask_radius=2.0)
```

The code below retains the original computation and records every assertion before displaying the reconstruction.


In [ ]:
def test_reconstruct_LPRec3d_tomobar_i12_dataset1_autopad(i12_dataset1: tuple):
    force_clean_gpu_memory()
    projdata = i12_dataset1[0]
    angles = i12_dataset1[1]
    flats = i12_dataset1[2]
    darks = i12_dataset1[3]
    del i12_dataset1
    data_normalised = dark_flat_field_correction(projdata, flats, darks)
    data_normalised = minus_log(data_normalised)
    data_normalised_cut = data_normalised[:, 5:8, :]
    del flats, darks, projdata, data_normalised
    force_clean_gpu_memory()
    recon_data = LPRec3d_tomobar(
        data_normalised_cut,
        np.deg2rad(angles),
        center=1253.75,
        detector_pad=True,
        filter_type="shepp",
        filter_freq_cutoff=1.0,
        recon_mask_radius=2.0,
    )
    record_check("recon_data.flags.c_contiguous", recon_data.flags.c_contiguous)
    recon_data = cp.asnumpy(recon_data)
    record_check("int(np.sum(recon_data)) == 9147", int(np.sum(recon_data)) == 9147)
    record_check(
        "pytest.approx(np.max(recon_data), rel=0.001) == 0.0063818083",
        pytest.approx(np.max(recon_data), rel=0.001) == 0.0063818083,
    )
    record_check(
        "pytest.approx(np.min(recon_data), rel=0.001) == -0.006200762",
        pytest.approx(np.min(recon_data), rel=0.001) == -0.006200762,
    )
    record_check("recon_data.dtype == np.float32", recon_data.dtype == np.float32)
    record_check(
        "recon_data.shape == (2560, 3, 2560)", recon_data.shape == (2560, 3, 2560)
    )
    report_reconstruction(recon_data)

In [ ]:
current_case = "test_reconstruct_LPRec3d_tomobar_i12_dataset1_autopad"
test_reconstruct_LPRec3d_tomobar_i12_dataset1_autopad(load_dataset("i12_dataset1"))
force_clean_gpu_memory()
print(json.dumps([row for row in results if row["test"] == current_case], indent=2))

## LPRec_tomobar_i13_dataset1

Original case: `test_reconstruct_LPRec_tomobar_i13_dataset1`.

Method call (original arguments):

```python
LPRec3d_tomobar(data=stiched_data_180degrees, angles=np.deg2rad(angles[0:3000]), center=2560, detector_pad=False, filter_type='shepp', filter_freq_cutoff=1.0, recon_mask_radius=2.0)
```

The code below retains the original computation and records every assertion before displaying the reconstruction.


In [ ]:
def test_reconstruct_LPRec_tomobar_i13_dataset1(i13_dataset1: tuple):
    force_clean_gpu_memory()
    projdata = i13_dataset1[0]
    angles = i13_dataset1[1]
    flats = i13_dataset1[2]
    darks = i13_dataset1[3]
    del i13_dataset1
    data_normalised = dark_flat_field_correction(projdata, flats, darks)
    data_normalised = minus_log(data_normalised)
    del flats, darks, projdata
    force_clean_gpu_memory()
    stiched_data_180degrees = sino_360_to_180(
        data_normalised[:, 2:3, :], overlap=473.822265625, side="right"
    )
    del data_normalised
    force_clean_gpu_memory()
    recon_data = LPRec3d_tomobar(
        data=stiched_data_180degrees,
        angles=np.deg2rad(angles[0:3000]),
        center=2560,
        detector_pad=False,
        filter_type="shepp",
        filter_freq_cutoff=1.0,
        recon_mask_radius=2.0,
    )
    record_check("recon_data.flags.c_contiguous", recon_data.flags.c_contiguous)
    recon_data = cp.asnumpy(recon_data)
    record_check("int(np.sum(recon_data)) == 1142", int(np.sum(recon_data)) == 1142)
    record_check("recon_data.dtype == np.float32", recon_data.dtype == np.float32)
    record_check(
        "recon_data.shape == (5120, 1, 5120)", recon_data.shape == (5120, 1, 5120)
    )
    report_reconstruction(recon_data)

In [ ]:
current_case = "test_reconstruct_LPRec_tomobar_i13_dataset1"
test_reconstruct_LPRec_tomobar_i13_dataset1(load_dataset("i13_dataset1"))
force_clean_gpu_memory()
print(json.dumps([row for row in results if row["test"] == current_case], indent=2))

## LPRec3d_tomobar_i13_dataset2

Original case: `test_reconstruct_LPRec3d_tomobar_i13_dataset2`.

Method call (original arguments):

```python
LPRec3d_tomobar(data=data_normalised, angles=np.deg2rad(angles), center=1286.25, detector_pad=False, filter_type='shepp', filter_freq_cutoff=1.0, recon_mask_radius=2.0)
```

The code below retains the original computation and records every assertion before displaying the reconstruction.


In [ ]:
def test_reconstruct_LPRec3d_tomobar_i13_dataset2(i13_dataset2: tuple):
    force_clean_gpu_memory()
    projdata = i13_dataset2[0]
    angles = i13_dataset2[1]
    flats = i13_dataset2[2]
    darks = i13_dataset2[3]
    del i13_dataset2
    data_normalised = dark_flat_field_correction(projdata, flats, darks)
    data_normalised = minus_log(data_normalised)
    del flats, darks, projdata
    force_clean_gpu_memory()
    recon_data = LPRec3d_tomobar(
        data=data_normalised,
        angles=np.deg2rad(angles),
        center=1286.25,
        detector_pad=False,
        filter_type="shepp",
        filter_freq_cutoff=1.0,
        recon_mask_radius=2.0,
    )
    record_check("recon_data.flags.c_contiguous", recon_data.flags.c_contiguous)
    recon_data = cp.asnumpy(recon_data)
    record_check("int(np.sum(recon_data)) == 3448", int(np.sum(recon_data)) == 3448)
    record_check(
        "pytest.approx(np.max(recon_data), rel=0.001) == 0.010543354",
        pytest.approx(np.max(recon_data), rel=0.001) == 0.010543354,
    )
    record_check(
        "pytest.approx(np.min(recon_data), rel=0.001) == -0.008385599",
        pytest.approx(np.min(recon_data), rel=0.001) == -0.008385599,
    )
    record_check("recon_data.dtype == np.float32", recon_data.dtype == np.float32)
    record_check(
        "recon_data.shape == (2560, 10, 2560)", recon_data.shape == (2560, 10, 2560)
    )
    report_reconstruction(recon_data)

In [ ]:
current_case = "test_reconstruct_LPRec3d_tomobar_i13_dataset2"
test_reconstruct_LPRec3d_tomobar_i13_dataset2(load_dataset("i13_dataset2"))
force_clean_gpu_memory()
print(json.dumps([row for row in results if row["test"] == current_case], indent=2))

## LPRec3d_tomobar_k11_dataset2

Original case: `test_reconstruct_LPRec3d_tomobar_k11_dataset2`.

Method call (original arguments):

```python
LPRec3d_tomobar(data=data_normalised, angles=np.deg2rad(angles), center=1280.25, detector_pad=False, filter_type='shepp', filter_freq_cutoff=1.0, recon_mask_radius=2.0)
```

The code below retains the original computation and records every assertion before displaying the reconstruction.


In [ ]:
def test_reconstruct_LPRec3d_tomobar_k11_dataset2(k11_dataset2: tuple):
    force_clean_gpu_memory()
    projdata = k11_dataset2[0]
    angles = k11_dataset2[1]
    flats = k11_dataset2[2]
    darks = k11_dataset2[3]
    del k11_dataset2
    data_normalised = dark_flat_field_correction(projdata, flats, darks)
    data_normalised = minus_log(data_normalised)
    del flats, darks, projdata
    force_clean_gpu_memory()
    recon_data = LPRec3d_tomobar(
        data=data_normalised,
        angles=np.deg2rad(angles),
        center=1280.25,
        detector_pad=False,
        filter_type="shepp",
        filter_freq_cutoff=1.0,
        recon_mask_radius=2.0,
    )
    record_check("recon_data.flags.c_contiguous", recon_data.flags.c_contiguous)
    recon_data = recon_data.get()
    record_check(
        "isclose(np.sum(recon_data), 10865.37, abs_tol=10 ** (-2))",
        isclose(np.sum(recon_data), 10865.37, abs_tol=10 ** (-2)),
    )
    record_check("recon_data.dtype == np.float32", recon_data.dtype == np.float32)
    record_check(
        "recon_data.shape == (2560, 25, 2560)", recon_data.shape == (2560, 25, 2560)
    )
    report_reconstruction(recon_data)

In [ ]:
current_case = "test_reconstruct_LPRec3d_tomobar_k11_dataset2"
test_reconstruct_LPRec3d_tomobar_k11_dataset2(load_dataset("k11_dataset2"))
force_clean_gpu_memory()
print(json.dumps([row for row in results if row["test"] == current_case], indent=2))

In [ ]:
check_results()